# exp_009 — GRPO of Qwen3-4B-Thinking-2507

**Runtime:** Colab Pro — A100 80GB (40GB will OOM at 8 generations × 4096 tokens)  
**Goal:** Train Qwen3-4B-Thinking-2507 with GRPO using ground-truth correctness as reward, push merged float16 weights to HuggingFace.  
**Time estimate:** Pilot ~30 min; full run ~6-10 hrs per epoch.

### Before running
1. Add `HF_TOKEN` to Colab Secrets (left sidebar → key icon).
2. Cell 4 prompts you to upload `public.jsonl`, `dev.jsonl`, `judger.py`, `utils.py`.
3. Runtime → Change runtime type → A100 GPU (80GB if available).
4. **First run:** keep `PILOT_MODE = True`. Verify mean reward trends upward before doing a full run.

In [ ]:
# Cell 1 — Install dependencies
#
# CRITICAL: Before running, do Runtime → "Disconnect and delete runtime"
# (NOT just "Restart session") if a previous install attempt failed.
#
# After this cell completes: Runtime → Restart session, then run from Cell 2.

# Colab Pro = CUDA 12.8 + Python 3.12. Latest vllm (0.20+) ships cu13 wheels
# which break here. Pin to vllm 0.11.0 — last release before the cu13 cutover,
# its default PyPI wheel is cu128-compatible.
!pip install --no-deps vllm==0.11.0

# Verify vllm actually installed
!pip show vllm | head -3

# unsloth + unsloth_zoo without dep changes
!pip install -q --no-deps unsloth
!pip install -q --upgrade --no-cache-dir --no-deps unsloth_zoo

# vllm's lightweight Python-only runtime deps (skipped by --no-deps)
!pip install -q msgspec pyzmq tiktoken partial-json-parser openai prometheus_client lm-format-enforcer outlines diskcache py-cpuinfo

# Training-stack deps (bitsandbytes is required for 4-bit QLoRA on the policy model)
!pip install -q trl==0.14.0 datasets bitsandbytes sympy antlr4-python3-runtime==4.11

print("\n✅ Installs done. NOW: Runtime → Restart session, then run from Cell 2.")

In [ ]:
# Cell 2 — Config
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

HF_REPO_ID = "TrevorDuong/qwen3-4b-thinking-grpo"  # private repo

# Use the Thinking variant — competition mandates this exact model.
# exp_008 mistakenly trained from Qwen/Qwen3-4B (non-thinking) and lost reasoning behavior.
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

# ------------------------- Pilot vs Full -------------------------
# PILOT_MODE: 50 prompts, 1 epoch, ~30 min. Confirms mean reward trends up.
# Full mode: ~926 prompts, 1 epoch, several hours. Only run after pilot succeeds.
PILOT_MODE = True
PILOT_N    = 50

# ------------------------- LoRA -------------------------
MAX_SEQ_LENGTH      = 5120   # prompt (~1024) + completion (4096) + headroom
MAX_PROMPT_LENGTH   = 1024
MAX_COMPLETION_LEN  = 4096
LORA_R              = 16
LORA_ALPHA          = 32

# ------------------------- GRPO -------------------------
NUM_GENERATIONS     = 8       # 8 samples per prompt; sparse-reward mitigation
TRAIN_BATCH_SIZE    = 1       # per-device prompt batch (each prompt -> 8 generations)
GRAD_ACCUM_STEPS    = 4       # effective prompt batch = 4
NUM_EPOCHS          = 1
LEARNING_RATE       = 5e-6    # lower than SFT — RL is unstable at SFT-scale LRs
BETA                = 0.04    # KL penalty to base model
TEMPERATURE         = 1.0     # higher than inference temp — encourages exploration

# ------------------------- Sampling for inference (matches exp_004) -------------------------
INFER_TEMPERATURE   = 0.6
INFER_TOP_P         = 0.95
INFER_TOP_K         = 20

RANDOM_SEED         = 42

print(f"PILOT_MODE: {PILOT_MODE}  ({'50 prompts' if PILOT_MODE else 'full ~926 prompts'})")
print(f"Model: {MODEL_NAME}")

In [ ]:
# Cell 3 — Smoke load: verify Thinking-2507 loads + chat template works.
# 5-minute fail-fast check — saves you from learning at hour 4 that the base is wrong.
from unsloth import FastLanguageModel
import torch, gc

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    fast_inference=True,           # vLLM backend for fast GRPO sampling
    gpu_memory_utilization=0.6,    # leave headroom for LoRA + KV during training
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    use_gradient_checkpointing="unsloth",   # MUST be on — 4096-token completions OOM otherwise
    random_state=RANDOM_SEED,
)

# Verify chat template applies cleanly with a thinking-style turn
test_msgs = [
    {"role": "system", "content": "You are an expert mathematician."},
    {"role": "user",   "content": "What is 7 × 8?"},
]
rendered = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print("=== Inference prompt ===")
print(rendered[-500:])
print("\n\nSmoke load OK. Model loaded:", MODEL_NAME)
model.print_trainable_parameters()

In [ ]:
# Cell 4 — Upload competition files
# Need: public.jsonl (questions+answers), dev.jsonl (200 held-out), judger.py, utils.py.
import os
from google.colab import files

REQUIRED = [
    ("/content/public.jsonl",  "data/public.jsonl"),
    ("/content/dev.jsonl",     "data/splits/dev.jsonl"),
    ("/content/judger.py",     "judger.py (repo root)"),
    ("/content/utils.py",      "utils.py (repo root)"),
]

for path, label in REQUIRED:
    if not os.path.exists(path):
        print(f"Upload: {label}")
        files.upload()
    else:
        print(f"Already present: {path}")

import sys
sys.path.insert(0, "/content")
from judger import Judger
JUDGER = Judger()
print("Judger loaded.")

In [ ]:
# Cell 5 — Prompts (exp_004 config — best found prior to fine-tuning)
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FEWSHOT_MATH = []

FEWSHOT_MCQ = [
    {"role": "user", "content": (
        "Find 1 over 6 + 1 over 8.\n\n"
        "Options:\nA. 7 over 24\nB. 2 over 14\nC. 1 over 4\nD. 7 over 48\n"
        "E. 2 over 24\nF. 1 over 14\nG. 1 over 2\nH. 8 over 14\nI. 0.21 Repeating\nJ. 4 over 24"
    )},
    {"role": "assistant", "content": "Common denominator is 24: 1/6 = 4/24 and 1/8 = 3/24, so 4/24 + 3/24 = 7/24. \\boxed{A}"},
    {"role": "user", "content": (
        "The function value of $\\cos(\\pi + 5i)$ is ( ).\n\n"
        "Options:\nA. -cosh5\nB. -sinh5\nC. sin5i\nD. -sin5\nE. cos5\n"
        "F. cosh5i\nG. sinh5\nH. -cos5\nI. cosh5\nJ. -sin5i"
    )},
    {"role": "assistant", "content": "$\\cos(\\pi + 5i) = -\\cos(5i) = -\\cosh(5)$, using $\\cos(\\pi+x) = -\\cos x$ and $\\cos(ix) = \\cosh x$. \\boxed{A}"},
    {"role": "user", "content": (
        "Find the range of $f(x) = \\frac{ x }{ 1+x^2 }$.\n\n"
        "Options:\nA. -\\frac{1}{3} \\le y \\le \\frac{1}{3}\nB. -\\frac{1}{\\sqrt{3}} \\le y \\le \\frac{1}{\\sqrt{3}}\n"
        "C. -\\frac{1}{4} \\le y \\le \\frac{1}{4}\nD. -\\frac{1}{\\sqrt{2}} \\le y \\le \\frac{1}{\\sqrt{2}}\n"
        "E. -\\frac{1}{\\sqrt{6}} \\le y \\le \\frac{1}{\\sqrt{6}}\nF. -\\frac{1}{2} \\le y \\le \\frac{1}{2}\n"
        "G. -\\frac{1}{\\sqrt{5}} \\le y \\le \\frac{1}{\\sqrt{5}}\nH. -1 \\le y \\le 1\n"
        "I. -\\frac{1}{\\sqrt{7}} \\le y \\le \\frac{1}{\\sqrt{7}}\nJ. -\\frac{1}{\\sqrt{4}} \\le y \\le \\frac{1}{\\sqrt{4}}"
    )},
    {"role": "assistant", "content": "$f'(x) = \\frac{1-x^2}{(1+x^2)^2} = 0$ at $x=\\pm 1$, giving $f(\\pm 1) = \\pm 1/2$. So range is $-1/2 \\le y \\le 1/2$. \\boxed{F}"},
]

In [ ]:
# Cell 6 — Build training dataset (public minus dev split, with answer/options as columns)
import json, random
from datasets import Dataset

random.seed(RANDOM_SEED)
LETTERS = "ABCDEFGHIJ"

# Load held-out dev IDs (must NOT appear in training data — leakage)
dev_ids = set()
with open("/content/dev.jsonl") as f:
    for line in f:
        dev_ids.add(json.loads(line)["id"])
print(f"Dev IDs (excluded): {len(dev_ids)}")

rows = []
with open("/content/public.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in dev_ids:
            continue
        is_mcq = "options" in ex and ex["options"]
        question_text = ex["question"]
        if is_mcq:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(ex["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"
            sys_prompt = SYSTEM_PROMPT_MCQ
            fewshots = FEWSHOT_MCQ
        else:
            sys_prompt = SYSTEM_PROMPT_MATH
            fewshots = FEWSHOT_MATH

        msgs = [{"role": "system", "content": sys_prompt}]
        msgs.extend(fewshots)
        msgs.append({"role": "user", "content": question_text})

        rows.append({
            "prompt": msgs,
            "answer": ex["answer"],     # list of gold strings — passed as kwarg to reward fn
            "options": ex.get("options", []),
            "is_mcq": is_mcq,
            "id": ex["id"],
        })

random.shuffle(rows)
if PILOT_MODE:
    rows = rows[:PILOT_N]
    print(f"PILOT_MODE — using {len(rows)} prompts")
else:
    print(f"Full mode — {len(rows)} prompts")

train_dataset = Dataset.from_list(rows)
print(train_dataset)
print("\nMCQ count:", sum(r["is_mcq"] for r in rows), "  Free-form:", sum(not r["is_mcq"] for r in rows))

In [ ]:
# Cell 7 — Reward functions
# Two reward signals:
#   1. correctness_reward: +1.0 if Judger.auto_judge passes, else 0.0
#   2. format_reward:     +0.1 if response contains \boxed{} after </think>
# Format bonus ensures non-zero gradient on uniformly-wrong batches (sparse-reward mitigation).
import re
import signal

def extract_post_think(text: str) -> str:
    """Return whatever follows the last </think>. If no </think>, return text.
    Qwen3-Thinking models emit reasoning inside <think>...</think> then the answer.
    Reward should ignore \\boxed{} that appears INSIDE thinking.
    """
    idx = text.rfind("</think>")
    if idx == -1:
        return text
    return text[idx + len("</think>"):]


def correctness_reward(prompts, completions, answer=None, options=None, is_mcq=None, **kwargs):
    """GRPO reward signature: receives lists, returns list[float].
    Each completion is a list of {role, content} message dicts (one per turn).
    """
    rewards = []
    for i, comp in enumerate(completions):
        # comp is list of message dicts — concat assistant content
        if isinstance(comp, list):
            text = "".join(m.get("content", "") for m in comp)
        else:
            text = str(comp)
        post = extract_post_think(text)
        gold = answer[i] if answer else []
        opts = options[i] if options else []
        try:
            # auto_judge needs options as list of None/list (one per gold). Match shape.
            opts_list = [opts] if gold and len(gold) > 0 else [None]
            opts_list = opts_list * len(gold) if gold else [None]
            ok = JUDGER.auto_judge(post, gold, opts_list)
        except Exception:
            ok = False
        rewards.append(1.0 if ok else 0.0)
    return rewards


_BOXED_RE = re.compile(r"\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}")

def format_reward(prompts, completions, **kwargs):
    """+0.1 if the response (after </think>) contains \\boxed{}."""
    rewards = []
    for comp in completions:
        if isinstance(comp, list):
            text = "".join(m.get("content", "") for m in comp)
        else:
            text = str(comp)
        post = extract_post_think(text)
        rewards.append(0.1 if _BOXED_RE.search(post) else 0.0)
    return rewards


# Quick self-test on a fake completion
_test_comp = [{"role": "assistant", "content": "<think>let me reason \\boxed{42} no wait</think>\nFinal: \\boxed{42}"}]
_post = extract_post_think(_test_comp[0]["content"])
assert "<think>" not in _post
assert "\\boxed{42}" in _post
print("Reward helpers self-test OK.")

In [ ]:
# Cell 8 — GRPO training
import shutil, os, gc
from trl import GRPOConfig, GRPOTrainer

if os.path.exists("/content/exp009_checkpoints"):
    shutil.rmtree("/content/exp009_checkpoints")

gc.collect(); torch.cuda.empty_cache()

training_args = GRPOConfig(
    output_dir="/content/exp009_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    bf16=True,
    logging_steps=1,            # log every step — we want to see reward trend
    save_strategy="epoch",
    save_total_limit=1,
    seed=RANDOM_SEED,
    report_to="none",
    # ---- GRPO-specific ----
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LEN,
    temperature=TEMPERATURE,
    beta=BETA,                  # KL coefficient against base policy
    use_vllm=True,              # generation backend; unsloth wires this in
    vllm_gpu_memory_utilization=0.55,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward, format_reward],
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting GRPO training...")
print(f"Steps/epoch: ~{len(train_dataset) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print(f"Tokens/epoch: ~{len(train_dataset) * NUM_GENERATIONS * MAX_COMPLETION_LEN / 1e6:.1f}M\n")

trainer.train()
print("\nTraining complete. Inspect logged reward — it should trend upward.")

In [ ]:
# Cell 9 — Evaluate on dev split
# Run policy inference on the 200 held-out dev questions; score with Judger.
# Compare to exp_004 baseline (55.33% on full public set; expect similar on dev).
from vllm import SamplingParams

FastLanguageModel.for_inference(model)

dev_rows = []
with open("/content/dev.jsonl") as f:
    for line in f:
        dev_rows.append(json.loads(line))

# Build prompts the same way as training
dev_prompts = []
for ex in dev_rows:
    is_mcq = "options" in ex and ex["options"]
    qt = ex["question"]
    if is_mcq:
        opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(ex["options"]))
        qt = f"{qt}\n\nOptions:\n{opts}"
        msgs = [{"role": "system", "content": SYSTEM_PROMPT_MCQ}] + FEWSHOT_MCQ + [{"role": "user", "content": qt}]
    else:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT_MATH}] + FEWSHOT_MATH + [{"role": "user", "content": qt}]
    dev_prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

sampling = SamplingParams(
    temperature=INFER_TEMPERATURE,
    top_p=INFER_TOP_P,
    top_k=INFER_TOP_K,
    max_tokens=8192,
)

outputs = model.fast_generate(dev_prompts, sampling_params=sampling)

# Score
correct = 0; mcq_c = 0; mcq_n = 0; ff_c = 0; ff_n = 0
results = []
for ex, out in zip(dev_rows, outputs):
    text = out.outputs[0].text
    post = extract_post_think(text)
    gold = ex["answer"]
    opts = ex.get("options", [])
    is_mcq = bool(opts)
    opts_list = ([opts] * len(gold)) if gold else [None]
    try:
        ok = JUDGER.auto_judge(post, gold, opts_list)
    except Exception:
        ok = False
    if ok:
        correct += 1
        if is_mcq: mcq_c += 1
        else: ff_c += 1
    if is_mcq: mcq_n += 1
    else: ff_n += 1
    results.append({"id": ex["id"], "response": text, "correct": ok, "is_mcq": is_mcq})

print(f"Dev overall:    {correct}/{len(dev_rows)}  ({100*correct/len(dev_rows):.2f}%)")
print(f"Dev MCQ:        {mcq_c}/{mcq_n}  ({100*mcq_c/mcq_n:.2f}%)")
print(f"Dev free-form:  {ff_c}/{ff_n}  ({100*ff_c/ff_n:.2f}%)")
print("Baseline (exp_004): 55.33% / MCQ 63.20% / FF 51.40%")

with open("/content/dev_responses.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print("\nSaved /content/dev_responses.jsonl — download via Files panel.")

In [ ]:
# Cell 10 — Merge LoRA into base, save as float16 (only after pilot/full succeeds)
MERGED_PATH = "/content/exp009_merged"

print("Merging adapter into base weights (float16)...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {MERGED_PATH}")
!du -sh {MERGED_PATH}

In [ ]:
# Cell 11 — Push to HuggingFace (private repo)
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
print(f"Repo: https://huggingface.co/{HF_REPO_ID}")

print("Uploading merged model (~8GB)...")
api.upload_folder(
    folder_path=MERGED_PATH,
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Done. Set HF_REPO_ID = '{HF_REPO_ID}' in the Kaggle inference notebook.")